In [ ]:
import requests
import os
import h5py
import time
import matplotlib.colors as mcolors
import astropy.units as u
import sympy as sp
import sys
import numpy as np 
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from copy import deepcopy
from ipywidgets import IntProgress
from IPython.display import display

sys.path.append('./')
from mcmc import *
from parallel import *
from Prior.fit_prior import read_prior_par

from mpl_toolkits.axes_grid1 import make_axes_locatable
from astropy.cosmology import FlatLambdaCDM
from pysr import PySRRegressor
from scipy.stats import binned_statistic_dd
from sklearn.metrics import mean_squared_error, median_absolute_error
from scipy.optimize import curve_fit
from sklearn.mixture import GaussianMixture
from scipy.stats import norm


In [12]:
# get func

# define function to get data from api and downloads
baseURL = 'http://www.tng-project.org/api/'
headers = {"api-key": "96ec8ab7a20d9b1666d6f0a5a4a8e0b9"}

def get(path, local_filename=None, params=None, max_retries=5):
    if local_filename and os.path.exists(local_filename):
        print(f"File {local_filename} already exists. Skipping network request.")
        return local_filename

    url = path if path.startswith('http') else baseURL + path

    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=headers, stream=True)
            r.raise_for_status()

            if 'application/json' in r.headers.get('content-type', ''):
                return r.json()

            if not local_filename:
                if 'content-disposition' in r.headers:
                    local_filename = r.headers['content-disposition'].split("filename=")[1]
                else:
                    local_filename = url.split('/')[-1] if url.split('/')[-1] else 'downloaded_tng_file'

            if os.path.exists(local_filename):
                print(f"File {local_filename} derived from header already exists. Skipping save.")
                return local_filename

            # disk stream download routine with live visual readout
            total_size = int(r.headers.get('content-length', 0))
            block_size = 4 * 1024 * 1024  # 4 mb chunks
            
            if total_size > 0:
                print(f"Downloading {local_filename} ({total_size / (1024**3):.2f} GB)...")
            else:
                print(f"Downloading {local_filename} (Unknown total file size)...")

            with open(local_filename, 'wb') as f:
                dl = 0
                start_time = time.time()
                for chunk in r.iter_content(chunk_size=block_size):
                    if chunk:
                        f.write(chunk)
                        dl += len(chunk)
                        
                        # status bar tracker
                        if total_size > 0:
                            done = int(50 * dl / total_size)
                            percent = (dl / total_size) * 100
                            elapsed = time.time() - start_time
                            speed = (dl / elapsed) / (1024**2) if elapsed > 0 else 0
                            print(f"\r[{'=' * done}{' ' * (50-done)}] {percent:.1f}% ({speed:.1f} MB/s)", end="")
                        else:
                            print(f"\rDownloaded: {dl / (1024**2):.1f} MB...", end="")
            print("\nDownload Complete!")
            return local_filename

        except requests.exceptions.HTTPError as e:
            # check for standard server-side rate limits or transient auth failures
            if r.status_code == 403 and attempt < max_retries - 1:
                wait_time = 2 ** attempt 
                print(f"\nAttempt {attempt+1} hit 403. Retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue
            else:
                raise e

In [13]:
# initialize all data and parameters

snap = get("http://www.tng-project.org/api/TNG300-1/snapshots/")
tree_data = {'subhalos': {}}
with h5py.File(r"C:\Users\rizkw\Documents\LCP Research\Project Files\Reproducing Florine's Work\TNG300_1_data.hdf5","r") as f:
    subhalos_grp = f['subhalos']
    for key in subhalos_grp.keys():
        tree_data['subhalos'][key] = subhalos_grp[key][()]

with h5py.File(r"C:\Users\rizkw\Documents\LCP Research\Project Files\Reproducing Florine's Work\TNG-Cluster_data.hdf5",'r') as f:
    subhalos_grp = f['subhalos']
    for key in subhalos_grp.keys():
        new_data = subhalos_grp[key][()]
        existing_data = tree_data['subhalos'][key]
        tree_data['subhalos'][key] = np.concatenate((existing_data, new_data), axis=0)

cosmology = FlatLambdaCDM(H0=67.74, Om0=0.3089, Ob0=0.0486)

lookback_infall3 = cosmology.lookback_time(tree_data['subhalos']['infall_z3']).value
lookback_infall1 = cosmology.lookback_time(tree_data['subhalos']['infall_z1']).value
lookback_sub = cosmology.lookback_time(tree_data['subhalos']['z']).value

tree_data['subhalos']['lookback_time3'] = lookback_infall3 - lookback_sub

tree_data['subhalos']['lookback_time1'] = lookback_infall1 - lookback_sub

tree_data['subhalos']['norm_r_3d'] = np.sqrt((tree_data['subhalos']['norm_r_xy']**2 + tree_data['subhalos']['norm_r_xz']**2 
                                              + tree_data['subhalos']['norm_r_yz']**2) / 2)

r_axis = {}
v_axis = {}
infall_axis3 = {}
infall_axis1 = {}
stel = {}
vir = {}
log10stel = {}
log10vir = {}
steps = {}
z_inf = {}
infall_axis1_clean = {}
r_axis_clean = {}
v_axis_clean = {}
m_axis_clean = {}

snapshots = [99, 84, 78, 67, 59, 50]
redshifts = [2.22044604925031e-16, 0.19728418237601, 0.297717684517447, 0.503047523244883, 0.700106353718523, 0.99729422578194]
for snapshot in snapshots:
    snapRmask = (tree_data['subhalos']['snap'] == snapshot) & (tree_data['subhalos']['norm_r_3d'] <= 1.0)
    r_axis[snapshot] = []
    v_axis[snapshot] = []
    infall_axis3[snapshot] = []
    infall_axis1[snapshot] = []
    stel[snapshot] = []
    vir[snapshot] = []
    log10stel[snapshot] = []
    log10vir[snapshot] = []
    steps[snapshot] = []
    z_inf[snapshot] = []

    r_axis[snapshot].append(tree_data['subhalos']['norm_r_xy'][snapRmask])
    r_axis[snapshot].append(tree_data['subhalos']['norm_r_xz'][snapRmask])
    r_axis[snapshot].append(tree_data['subhalos']['norm_r_yz'][snapRmask])
    v_axis[snapshot].append(tree_data['subhalos']['norm_v_z'][snapRmask])
    v_axis[snapshot].append(tree_data['subhalos']['norm_v_y'][snapRmask])
    v_axis[snapshot].append(tree_data['subhalos']['norm_v_x'][snapRmask])
    infall_axis3[snapshot].append(tree_data['subhalos']['lookback_time3'][snapRmask])
    infall_axis3[snapshot].append(tree_data['subhalos']['lookback_time3'][snapRmask])
    infall_axis3[snapshot].append(tree_data['subhalos']['lookback_time3'][snapRmask])
    infall_axis1[snapshot].append(tree_data['subhalos']['lookback_time1'][snapRmask])
    infall_axis1[snapshot].append(tree_data['subhalos']['lookback_time1'][snapRmask])
    infall_axis1[snapshot].append(tree_data['subhalos']['lookback_time1'][snapRmask])
    stel[snapshot].append(tree_data['subhalos']['stel_mass'][snapRmask] * 1e10/ 0.6774)
    stel[snapshot].append(tree_data['subhalos']['stel_mass'][snapRmask] * 1e10/ 0.6774)
    stel[snapshot].append(tree_data['subhalos']['stel_mass'][snapRmask] * 1e10/ 0.6774)
    vir[snapshot].append(tree_data['subhalos']['vir_mass'][snapRmask] * 1e10/ 0.6774)
    vir[snapshot].append(tree_data['subhalos']['vir_mass'][snapRmask] * 1e10/ 0.6774)
    vir[snapshot].append(tree_data['subhalos']['vir_mass'][snapRmask] * 1e10/ 0.6774)
    steps[snapshot].append(tree_data['subhalos']['step_counts3'][snapRmask])
    steps[snapshot].append(tree_data['subhalos']['step_counts3'][snapRmask])
    steps[snapshot].append(tree_data['subhalos']['step_counts3'][snapRmask])
    z_inf[snapshot].append(tree_data['subhalos']['infall_z3'][snapRmask])
    z_inf[snapshot].append(tree_data['subhalos']['infall_z3'][snapRmask])
    z_inf[snapshot].append(tree_data['subhalos']['infall_z3'][snapRmask])
    r_axis[snapshot] = np.concatenate(r_axis[snapshot])
    v_axis[snapshot] = np.concatenate(v_axis[snapshot])
    infall_axis3[snapshot] = np.concatenate(infall_axis3[snapshot])
    infall_axis1[snapshot] = np.concatenate(infall_axis1[snapshot])
    log10stel[snapshot] = np.log10(np.concatenate(stel[snapshot]))
    log10vir[snapshot] = np.log10(np.concatenate(vir[snapshot]))
    steps[snapshot] = np.concatenate(steps[snapshot])
    z_inf[snapshot] = np.concatenate(z_inf[snapshot])

    # filter nans
    valid = ~np.isnan(infall_axis1[snapshot])
    infall_axis1_clean[snapshot] = infall_axis1[snapshot][valid]
    r_axis_clean[snapshot] = r_axis[snapshot][valid]
    v_axis_clean[snapshot] = v_axis[snapshot][valid]
    m_axis_clean[snapshot] = log10stel[snapshot][valid]

In [15]:
# bin data 

X_binned = {}
y_mean = {}
y_q1 = {}
y_med = {}
y_q3 = {}

for snapshot in snapshots:
    # prepare all raw inputs as a matrix with columns of each input
    X_clean = np.column_stack((r_axis_clean[snapshot], np.abs(v_axis_clean[snapshot]), m_axis_clean[snapshot])) #, log10vir[snapshot]))
    y_clean = infall_axis1_clean[snapshot]

    # X_clean[np.isinf(X_raw)] = np.nan

    # clean_rows = ~np.isnan(X_raw).any(axis=1) & ~np.isnan(y_raw)

    # X_clean = X_raw[clean_rows]
    # y_clean = y_raw[clean_rows]
    # create percentile bins

    r_edges = np.percentile(X_clean[:,0], np.linspace(0, 100, 12))
    v_edges = np.percentile(X_clean[:,1], np.linspace(0, 100, 12))
    m_edges = np.percentile(X_clean[:,2], np.linspace(0, 100, 9))
    # vir_edges = np.percentile(X_clean[:,3], np.linspace(0, 100, 5))

    quartile_bins = [r_edges, v_edges, m_edges] #, vir_edges]
    # sort data into bins

    # compute num of galaxies in each bin
    counts, _, _ = binned_statistic_dd(
        X_clean, values=None, statistic='count', bins=quartile_bins
    )

    # mask for only filled bins
    filled = np.where(counts > 0)
    Ngal_per = counts[filled]
    weights = np.log10(Ngal_per + 10)

    # extract mean features for grid size
    binned = []
    for i in range(X_clean.shape[1]):
        means, _, _ = binned_statistic_dd(X_clean, values=X_clean[:,i], statistic='mean', bins=quartile_bins)
        binned.append(means[filled])
    X_binned[snapshot] = np.column_stack(binned)

    mean_y, _, _ = binned_statistic_dd(X_clean, values=y_clean, statistic='mean', bins=quartile_bins)
    y_mean[snapshot] = mean_y[filled]

    q1_y_grid, _, _ = binned_statistic_dd(X_clean, values=y_clean, statistic=lambda x: np.percentile(x, 25), bins=quartile_bins)
    y_q1[snapshot] = q1_y_grid[filled]

    q2_y_grid, _, _ = binned_statistic_dd(X_clean, values=y_clean, statistic=lambda x: np.percentile(x, 50), bins=quartile_bins)
    y_med[snapshot] = q2_y_grid[filled]

    q3_y_grid, _, _ = binned_statistic_dd(X_clean, values=y_clean, statistic=lambda x: np.percentile(x, 75), bins=quartile_bins)
    y_q3[snapshot] = q3_y_grid[filled]

    print(f"Num of binned data points for fitting: {len(y_mean[snapshot])}")
    # mask for valid entries in bins
    mean_mask = ~np.isnan(y_mean[snapshot]) & (y_mean[snapshot] > 0)
    q1_mask = ~np.isnan(y_q1[snapshot]) & (y_q1[snapshot] > 0)
    med_mask = ~np.isnan(y_med[snapshot]) & (y_med[snapshot] > 0)
    q3_mask = ~np.isnan(y_q3[snapshot]) & (y_q3[snapshot] > 0)

    y_mean[snapshot] = y_mean[snapshot][mean_mask]
    y_q1[snapshot] = y_q1[snapshot][q1_mask]
    y_med[snapshot] = y_med[snapshot][med_mask]
    y_q3[snapshot] = y_q3[snapshot][q3_mask]

    # use if want to fit in log M* or log 10^10 M*
    # X_binned[snapshot][:,2] = X_binned[snapshot][:,2]-10
    # X_binned[snapshot][:,3] = X_binned[snapshot][:,3]-10

Num of binned data points for fitting: 968
Num of binned data points for fitting: 968
Num of binned data points for fitting: 968
Num of binned data points for fitting: 968
Num of binned data points for fitting: 968
Num of binned data points for fitting: 968


In [19]:
# initialize bayesian machine

prior_par = read_prior_par(r"C:\Users\rizkw\Documents\LCP Research\Project Files\Tinf_at_R200_Project\Prior\final_prior_param_sq.named_equations.nv13.np13.2016-09-01 17:05:57.196882.dat")

Ts = [1] + [1.04**k for k in range(1, 20)]

pms = Parallel(
    Ts,
    variables=['r', 'v', 'logms'],
    parameters=['a%d' % i for i in range(13)],
    x=X_binned[99], y=y_mean[99],
    prior_par=prior_par,
)

OSError: [Errno 22] Invalid argument: 'C:\\Users\\rizkw\\Documents\\LCP Research\\Project Files\\Tinf_at_R200_Project\\Prior\\final_prior_param_sq.named_equations.nv13.np13.2016-09-01 17:05:57.196882.dat'